<a href="https://colab.research.google.com/github/seongho-12/LG-hackathon-LLM-Lightning-/blob/basic-code/LG(LLM_lighting_hackathon).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 기존 패키지 삭제 후 깨끗하게 재설치
!pip uninstall -y llmcompressor
!pip install llmcompressor[torch] --extra-index-url https://download.pytorch.org/whl/cu121

Found existing installation: llmcompressor 0.9.0.1
Uninstalling llmcompressor-0.9.0.1:
  Successfully uninstalled llmcompressor-0.9.0.1
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
  Using cached llmcompressor-0.9.0.1-py3-none-any.whl.metadata (12 kB)
Using cached llmcompressor-0.9.0.1-py3-none-any.whl (282 kB)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -al /content/drive/MyDrive/hackathon/base_model

total 1334
drwx------ 2 root root    4096 Feb 13 07:17 assets
-rw------- 1 root root    5487 Jan 28 00:12 chat_template.jinja
-rw------- 1 root root    1527 Jan 28 00:12 config.json
-rw------- 1 root root     134 Jan 28 00:12 generation_config.json
drwx------ 2 root root    4096 Feb 13 07:17 .git
-rw------- 1 root root    1561 Jan 28 00:12 .gitattributes
-rw------- 1 root root   13288 Jan 28 00:12 LICENSE
-rw------- 1 root root 1219196 Jan 28 00:12 merges.txt
-rw------- 1 root root   37088 Jan 28 00:12 README.md
-rw------- 1 root root    6704 Jan 28 00:12 special_tokens_map.json
-rw------- 1 root root   70315 Jan 28 00:12 tokenizer_config.json


In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

MODEL_ID = "/content/drive/MyDrive/hackathon/base_model"
OUT_DIR  = "./model"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

print("[INFO] 모델/토크나이저 로드 완료")

print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")


print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")

os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

zip_name = "baseline_submit"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")



[INFO] 모델 로드 중...


`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델/토크나이저 로드 완료
[INFO] 캘리브레이션 데이터 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

...

(31/31): Propagating: 100%|██████████| 256/256 [00:00<00:00, 749.73it/s]

2026-02-13T07:40:20.486275+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


2026-02-13T07:40:20.616033+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[INFO] GPTQ 완료
2026-02-13T07:40:20.686370+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:09, 22.34it/s]


[INFO] 모델 저장 완료: ./model
[INFO] baseline_submit.zip 생성 중...
[INFO] 생성 완료: baseline_submit.zip
